In [1]:
import re

import json
import os

from collections import Counter

import networkx as nx

# NATURAL LANGUAGE TOOLKIT
# pip install nltk
import nltk

# Download WordNet data needed for Lemmatization
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

#Download stopwords
from nltk.corpus import stopwords
nltk.download('stopwords')

import math
import random

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\computer-1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\computer-1\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\computer-1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
class Book(object):

    def __init__(self, author, title, book):

        author = author.upper()
        author = author.replace(" ", "-")
        self.author = author

        title = title.lower()
        title = title.replace(" ", "-")
        self.title = title

        self.path = author + "_" + title

        self.create_metadata()

        self.book = book
        self.text = self.curate_book()

        self.sentences_list()
        self.words_list()

        self.word_connections()

        self.create_graph()

        self.word_count()

        #self.add_to_theka()

        

    

    def create_metadata(self):

        data = {
                "author" : self.author, 
                "title" : self.title
            }
        os.makedirs("./theka/" + self.path, exist_ok=True)
        file_path = "./theka/" + self.path + "/data.json"
        with open(file_path, "w") as file:
            json.dump(data, file, indent=4)



    def add_metadata(self, key, value):
        
        file_path = "./theka/" + self.path + "/data.json"
        with open(file_path, 'r') as file:
            data = json.load(file)
        
        data[key] = value

        with open(file_path, 'w') as file:
            json.dump(data, file, indent=4)

    def read_metadata(self, key):

        file_path = "./theka/" + self.path + "/data.json"
        with open(file_path, 'r') as file:
            data = json.load(file)
        
        value = data[key]
        return value
    
    """
    def add_to_theka(self):
        file_path = "./theka/data.json"
        with open(file_path, "r") as file:
            data = json.load(file)
        data["books"].append(self.path)
        with open(file_path, "w") as file:
            json.dump(data, file, indent=4)
    """



    def curate_book(self):
        text = self.book
        text = text.replace("}", " ")
        text = text.replace("{", " ")
        text = text.replace("-", " ")
        text = text.replace("\n", " ") 
        text = text.replace(" .", ".")
        text = text.replace(" ,", ",")
        text = text.replace("\r", " ")
        text = re.sub(r'\s+', ' ', text)
        return text
    

    def sentences_list(self):

        all_sentences = self.text.split(". ")

        sentences = []
        for sentence in all_sentences:
            if len(sentence) > 3:
                sentence = sentence + "."
                sentences.append(sentence)
                
        self.add_metadata("sentences", sentences)


    def get_sentences(self):
        sentences = self.read_metadata("sentences")
        return sentences



    def words_list(self):

        lemmatizer = WordNetLemmatizer()

        all_words = re.findall(r'\b\w+\b', self.text.lower())

        words = []
        stop_words = set(stopwords.words("english"))
        for word in all_words:

            base_form = lemmatizer.lemmatize(word)

            if base_form not in stop_words and len(base_form)>2:
                words.append(base_form)

        self.add_metadata("words", words)


    def get_words(self):
        words = self.read_metadata("words")
        return words



    def word_connections(self):
        word_connections = []
        words = self.get_words()

        for i in range(len(words) - 1):
            word_connections.append((words[i], words[i+1]))
        self.word_connections = word_connections
        self.add_metadata("word_connections", word_connections)


    def word_count(self):
        words = self.get_words()
        words_count = Counter(words)
        self.add_metadata("words_count", words_count)

    def get_words_count(self):
        words_count = self.read_metadata("words_count")
        return words_count
    
    
    def create_graph(self):
        graph = nx.Graph()
        for word1, word2 in self.word_connections:
            graph.add_edge(word1, word2)
        self.eigenvector_centrality = nx.eigenvector_centrality(graph)
        self.add_metadata("eigenvector", self.eigenvector_centrality)

    def get_eigen_vector(self):
        eigen_vector = self.read_metadata("eigenvector")
        return eigen_vector
    

    def get_eigen_vector_of(self, word):
        vectors = self.get_eigen_vector()
        try:
            word_vector = vectors[word]
        except:
            word_vector = 0
        return word_vector

In [3]:
file_path = "books_to_analyze.json"


with open(file_path, 'r') as file:
        data = json.load(file)

flow_of_books = list(data.keys())

books = []

for bk in flow_of_books:
        file_path = "flow_of_books/" + data[bk]["file"]
        author = data[bk]["author"]
        title = bk

        with open(file_path, 'r') as file:
            book_content = file.read()

        book = Book(author, title, book_content)
        books.append(book)




In [4]:
#DICTIONARY FOR EIGENVECTOR

all_words = set()
list_of_dictionaries = []
for book in books:
    print(book)
    book_dictionary = book.get_eigen_vector()
    all_words.update(book_dictionary.keys())
    list_of_dictionaries.append(book_dictionary)

theka_dictionary = {}

for word in all_words:
    theka_dictionary[word] = [dictionary.get(word, 0) for dictionary in list_of_dictionaries]

sorted_dictionary = {key: theka_dictionary[key] for key in sorted(theka_dictionary)}

for key, value in sorted_dictionary.items():
    print(f"{key} ---> {value}")


000 ---> [0, 0.00019804945030674724, 0, 7.004190376093735e-05, 0]
004 ---> [0, 2.3593437533205863e-05, 0, 0, 0]
03758 ---> [0, 0, 0, 0.0011049457325445508, 0]
100 ---> [2.6870655032768303e-06, 0, 6.379639526327345e-06, 0, 0]
1001 ---> [0, 0, 0, 0, 5.2578226502277726e-05]
108 ---> [0.005432088552709602, 0, 0, 0, 0]
121 ---> [0, 0, 0.00026999529848787513, 0, 0]
123 ---> [3.085010200219177e-08, 0, 8.475038910828663e-08, 0, 0]
124 ---> [7.210365896957462e-10, 0, 2.0024004843621685e-09, 0, 0]
126 ---> [4.033210178125115e-09, 0, 4.7574524374433606e-09, 0, 0]
127 ---> [1.942312215667988e-07, 0, 2.100575458154854e-07, 0, 0]
1308 ---> [0, 0, 0, 3.871072197634132e-06, 0]
1311 ---> [0, 0, 0, 0, 6.6172364977687574e-06]
1314 ---> [2.807083870549041e-06, 0, 0, 0, 0]
1317 ---> [0, 0, 7.084970869895458e-06, 0, 0]
132 ---> [0, 5.939709604590632e-05, 0, 0, 0]
1320 ---> [0, 5.039982963933925e-06, 0, 0, 0]
180 ---> [8.937810147997402e-05, 0, 0, 0, 0]
1882 ---> [2.7396037115455388e-06, 0, 0, 0, 0]
192 --->

In [5]:
#DICTIONARY FOR NUMBER OF WORDS


all_words = set()
list_of_dictionaries = []
for book in books:
    book_dictionary = book.get_words_count()
    all_words.update(book_dictionary.keys())
    list_of_dictionaries.append(book_dictionary)

theka_dictionary = {}

for word in all_words:
    theka_dictionary[word] = [dictionary.get(word, 0) for dictionary in list_of_dictionaries]

sorted_dictionary_alphabetically = {key: theka_dictionary[key] for key in sorted(theka_dictionary)}


sorted_dictionary_by_numbers = dict(sorted(sorted_dictionary.items(), key=lambda item: sum(item[1]), reverse=True))

print(sorted_dictionary_by_numbers)

for key, value in sorted_dictionary_by_numbers.items():
    print(f"{key} ---> {value}")





{'said': [0.2749533731845322, 0.214612101068444, 0.28164523865227803, 0.2912199146569182, 0.2953872524462646], 'arthur': [0.22399891780380757, 0.18463585769821167, 0.23210386656375617, 0.2288233764189292, 0.1663114276375413], 'one': [0.20091163716375604, 0.17085492368734684, 0.18841032317702752, 0.1620533291330503, 0.1796157637603696], 'ford': [0.13108843146399266, 0.14818550524289745, 0.08642731153798525, 0.20735393066482366, 0.1864212245123338], 'would': [0.15610910708015052, 0.14383195126145829, 0.15705855056415408, 0.12127446477657965, 0.12416715324050286], 'know': [0.10323577834090623, 0.13744386476654613, 0.1414960698893625, 0.12627406501640775, 0.13850833973472487], 'like': [0.11849287835124953, 0.12927892668351867, 0.14899374551664946, 0.09513967496412201, 0.10748181200896857], 'thing': [0.12331057259931423, 0.14907617936541215, 0.12327581193295759, 0.09851360455209797, 0.10208594803571912], 'time': [0.11384549694236969, 0.12615407690881186, 0.14226178433073514, 0.1067688423929

In [6]:
#ASK TO THEKA

def ask_to_theka(question):
    for book in books:
        sentences_list = book.get_sentences()
        for sentence in sentences_list:
            if all(word in sentence.split() for word in question):
                print(f"{book.author}, {book.title}: \n\n\t\t {sentence}\n\n")



question = ["being"]
ask_to_theka(question)

ADAMS-DOUGLAS, life,-the-universe-and-everything: 

		 At least being lost in space kept you busy.


ADAMS-DOUGLAS, life,-the-universe-and-everything: 

		 He was stranded in prehistoric Earth as the result of a complex sequence of events which had involved him being alternately blown up and insulted in more bizarre regions of the Galaxy than he had ever dreamt existed, and though life had now turned very, very, very quiet, he was still feeling jumpy.


ADAMS-DOUGLAS, life,-the-universe-and-everything: 

		 When people protested to him, as they sometimes had done, that the plan was not merely misguided but actually impossible because of the number of people being born and dying all the time, he would merely fix them with a steely look and say, ‘A man can dream, can’t he?’ And so he had started out.


ADAMS-DOUGLAS, life,-the-universe-and-everything: 

		 ‘I took up being cruel to animals,’ he said airily.


ADAMS-DOUGLAS, life,-the-universe-and-everything: 

		 ‘I’m training it to like

# MACHINE LEARNING

In [7]:
class SOM(object):
    
    def __init__(self, grid_size, input_dim, learning_rate=0.1, radius=None, iterations=500):
        self.grid_size = grid_size  # Dimensions of the grid (e.g., (10, 10))
        self.input_dim = input_dim  # Dimensionality of input data
        self.learning_rate = learning_rate  # Initial learning rate
        self.radius = radius if radius else max(self.grid_size) / 2  # Initial radius of the neighborhood
        self.iterations = iterations  # Number of iteration

        # Initialize the weight matrix for the grid
        self.weights = self._initialize_weights()

    def _initialize_weights(self):
        """Initialize weights of the grid randomly"""
        weights = []
        for i in range(self.grid_size[0]):
            row = []
            for j in range(self.grid_size[1]):
                # Random weight vector for each neuron (within range [0, 1])
                row.append([random.random() for _ in range(self.input_dim)])
            weights.append(row)
        return weights
    
    def _euclidean_distance(self, v1, v2):
        """Compute Euclidean distance between two vectors"""
        return math.sqrt(sum((v1[i] - v2[i]) ** 2 for i in range(len(v1))))
    
    def _neighborhood_function(self, distance, radius):
        """Calculate the Gaussian neighborhood function"""
        return math.exp(-distance**2 / (2 * (radius**2)))
    
    def _find_best_matching_unit(self, x):
        """Find the Best Matching Unit (BMU)"""
        min_distance = float('inf')
        bmu_idx = (0, 0)
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                # Calculate the Euclidean distance between input x and the neuron weight
                distance = self._euclidean_distance(self.weights[i][j], x)
                if distance < min_distance:
                    min_distance = distance
                    bmu_idx = (i, j)
        return bmu_idx
    

    def _update_weights(self, x, bmu_idx, iteration):
        """Update the weights of the SOM neurons"""
        # Calculate the decay for learning rate and neighborhood size
        learning_rate = self.learning_rate * math.exp(-iteration / self.iterations)
        radius = self.radius * math.exp(-iteration / self.iterations)

        # Iterate over the grid and update weights based on their distance from BMU
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                # Calculate the distance from the BMU (Euclidean)
                distance = self._euclidean_distance([i, j], bmu_idx)
                if distance <= radius:
                    # Update the weights with the neighborhood function
                    influence = self._neighborhood_function(distance, radius)
                    for k in range(self.input_dim):
                        self.weights[i][j][k] += influence * learning_rate * (x[k] - self.weights[i][j][k])


    def train(self, data):
        """Train the SOM with the given data"""
        self.initial_radius = self.radius
        for iteration in range(self.iterations):
            self.radius = self.initial_radius * (1 - (iteration / self.iterations))
            print(self.radius)
            for x in data:
                # Find the Best Matching Unit (BMU)
                bmu_idx = self._find_best_matching_unit(x)
                # Update the weights of the SOM
                self._update_weights(x, bmu_idx, iteration)
                #self.radius -= self.learning_rate


    def plot(self):
        """Visualize the SOM using an ASCII art plot"""
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                # Print the dimension of the weight vector for each neuron
                print(f"[{self.weights[i][j][0]:.2f}, {self.weights[i][j][1]:.2f}, {self.weights[i][j][2]:.2f}]", end=" ")
            print()


    def place_values(self, dictionary):
        bmu_dict = {}
        for key in dictionary.keys():
            bmu_idx = self._find_best_matching_unit(dictionary[key])
            bmu_dict[key] = bmu_idx

        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                print_nothing = False
                for key in bmu_dict:
                    if bmu_dict[key] == (i, j):
                        dashes = ""
                        for l in range(10-len(key)):  dashes += "-"
                        print(key+dashes,  end=" ")
                        print_nothing = False
                        break
                    else:
                        print_nothing = True
                if print_nothing:
                    print("----------", end=" ")
            print()

In [8]:
first_100_items = dict(list(sorted_dictionary_by_numbers.items())[:200])
data = first_100_items
print(data)

{'said': [0.2749533731845322, 0.214612101068444, 0.28164523865227803, 0.2912199146569182, 0.2953872524462646], 'arthur': [0.22399891780380757, 0.18463585769821167, 0.23210386656375617, 0.2288233764189292, 0.1663114276375413], 'one': [0.20091163716375604, 0.17085492368734684, 0.18841032317702752, 0.1620533291330503, 0.1796157637603696], 'ford': [0.13108843146399266, 0.14818550524289745, 0.08642731153798525, 0.20735393066482366, 0.1864212245123338], 'would': [0.15610910708015052, 0.14383195126145829, 0.15705855056415408, 0.12127446477657965, 0.12416715324050286], 'know': [0.10323577834090623, 0.13744386476654613, 0.1414960698893625, 0.12627406501640775, 0.13850833973472487], 'like': [0.11849287835124953, 0.12927892668351867, 0.14899374551664946, 0.09513967496412201, 0.10748181200896857], 'thing': [0.12331057259931423, 0.14907617936541215, 0.12327581193295759, 0.09851360455209797, 0.10208594803571912], 'time': [0.11384549694236969, 0.12615407690881186, 0.14226178433073514, 0.1067688423929

In [9]:
# Normalize data

# Flatten all vectors and calculate the global Euclidean norm
all_values = [x for values in data.values() for x in values]
global_magnitude = math.sqrt(sum(x**2 for x in all_values))

# Normalize each vector using the global magnitude
normalized_data = {
    key: [x / global_magnitude for x in value]
    for key, value in data.items()
}

# Print the normalized dictionary
print(normalized_data)

{'said': [0.1405881379118191, 0.10973466269244889, 0.1440097977167904, 0.14890548550199872, 0.1510363131876826], 'arthur': [0.11453429497358689, 0.0944073212301418, 0.1186784872809391, 0.11700111924010488, 0.08503774168766418], 'one': [0.10272939235668373, 0.08736090521846096, 0.09633726690468722, 0.0828605065700949, 0.09184046543677803], 'ford': [0.0670276500624529, 0.07576966234794695, 0.04419169204261856, 0.10602344195028028, 0.09532020836071187], 'would': [0.07982112902010323, 0.0735436192903179, 0.08030659493718796, 0.0620096090537833, 0.06348868777899329], 'know': [0.05278613488071147, 0.07027728662184543, 0.07234924509995126, 0.06456598608556892, 0.07082157001032219], 'like': [0.0605873386104398, 0.06610242079654882, 0.0761829287638344, 0.04864646536183714, 0.054957201050913146], 'thing': [0.06305070414590944, 0.07622507853331795, 0.06303293045104964, 0.05037161051181421, 0.05219820791822847], 'time': [0.0582110568035535, 0.06450463421189269, 0.07274076736511918, 0.0545926480740

In [10]:
som = SOM(grid_size=(25, 25), input_dim=5, learning_rate=0.1, radius=25, iterations=100)
som.train(normalized_data.values())
print("- - - - - - - - - - - - - - - - ")

25.0
24.75
24.5
24.25
24.0
23.75
23.5
23.25
23.0
22.75
22.5
22.25
22.0
21.75
21.5
21.25
21.0
20.75
20.5
20.25
20.0
19.75
19.5
19.25
19.0
18.75
18.5
18.25
18.0
17.75
17.5
17.25
17.0
16.75
16.499999999999996
16.25
16.0
15.75
15.5
15.25
15.0
14.750000000000002
14.500000000000002
14.250000000000002
14.000000000000002
13.750000000000002
13.5
13.25
13.0
12.75
12.5
12.25
12.0
11.75
11.5
11.249999999999998
10.999999999999998
10.750000000000002
10.500000000000002
10.25
10.0
9.75
9.5
9.25
9.0
8.75
8.5
8.249999999999998
7.999999999999999
7.750000000000002
7.500000000000001
7.250000000000001
7.000000000000001
6.75
6.5
6.25
6.0
5.75
5.499999999999999
5.249999999999999
4.999999999999999
4.749999999999998
4.500000000000001
4.250000000000001
4.000000000000001
3.7500000000000004
3.5000000000000004
3.25
3.0
2.7499999999999996
2.4999999999999996
2.249999999999999
1.9999999999999991
1.7499999999999987
1.5000000000000013
1.250000000000001
1.0000000000000009
0.7500000000000007
0.5000000000000004
0.250000000

In [11]:
som.place_values(normalized_data)

said------ ---------- ---------- one------- ---------- would----- ---------- ---------- like------ ---------- ---------- could----- ---------- way------- ---------- something- ---------- ---------- ---------- little---- ---------- ---------- ---------- suddenly-- seemed---- 
---------- arthur---- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- head------ ---------- point----- 
---------- ---------- ---------- ---------- know------ ---------- ---------- thing----- ---------- ---------- ---------- ---------- say------- ---------- get------- ---------- ---------- ---------- people---- ---------- ---------- ---------- ---------- around---- ---------- 
---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- ---------- thought--- ---------- ---------- ---------- -------

In [13]:
print(books)

[<__main__.Book object at 0x000001E7C3287650>, <__main__.Book object at 0x000001E7FF0E0450>, <__main__.Book object at 0x000001E7C9FE3510>, <__main__.Book object at 0x000001E7C9DE4550>, <__main__.Book object at 0x000001E7FF146890>]


In [14]:
topic = "planet"
question = ["architecture"]





sorted_books_by_topic_centrality = {}
for book in books:
    vector = book.get_eigen_vector_of(topic)
    sorted_books_by_topic_centrality[book] = vector

sorted_books_by_topic_centrality = dict(sorted(sorted_books_by_topic_centrality.items(), key=lambda item: item[1], reverse=True))


for book in sorted_books_by_topic_centrality.keys():
    sentences_list = book.get_sentences()
    #print(f"{book.author}, {book.title}:")
    results = []
    for sentence in sentences_list:
        if all(word in sentence.split() for word in question):
            results.append(f"\n\n\t\t {sentence}\n\n")
    
    if len(results) != 0:
        print(f"{book.author}, {book.title}:")
        for r in results:
            print(r)



    

ADAMS-DOUGLAS, so-long,-and-thanks-for-all-the-fish:


		 A hatchway opened, crashed down through the Harrods Food Halls, demolished Harvey Nichols, and with a final grinding scream of tortured architecture toppled the Sheraton Park Tower.


